# Submit your predictions using (`mosqlient`)

To submit your predictions using `mosqlient` we will use the function 
`upload_prediction`. This function has the following parameters: 

| Field | Type | Description |
| :--- | :--- | :--- | 
| api_key | string | Your personal API authentication key.|
| repository | str | Model repository. Format: "{owner or org}/{name}" | 
| disease |  str | Disease code (ICD-10); "A90": Dengue, "A92.0": Chikungunya, "A92.5: Zika  |
| description | str or None | Prediction description |
| commit | str | Git commit hash to lastest version of Prediction's code in the Model's repository |
| case_definition | str | "reported" or "probable". The case definition used for the prediction data. |
| published | bool _(True)_ | Whether this prediction is visible to the public. |
| adm_level | int (0, 1, 2, 3) | Administrative level, options: 0, 1, 2, 3 (National, State, Municipal, Sub-municipal) |
| adm_0 | str _(BRA)_ | Country isocode. Default: "BRA" |
| adm_1 | int _(UF)_ | State geocode. Example: 33 for RJ |
| adm_2 | int _(IBGE)_ | City geocode. Example: 3304557 |
| adm_3 | int _(IBGE)_ | Sub-municipality geocode. |
| prediction | pd.DataFrame | pandas DataFrame containing the columns: "date", "pred", "lower_95","lower_90", "lower_80", "lower_50", "upper_50", "upper_80", "upper_90", "upper_95" |
 
**About the Location Data (The `adm` fields):** 

Fill only the geographic level that your predictions refer to; leave all other levels as null.
* State-level model: Set `adm_level = 1`, fill in `adm_1`, and set `adm_2` and `adm_3` to null.
* City-level model: Set `adm_level = 2`, fill in `adm_2`, and set `adm_1` and `adm_3` to null.

**Understanding the columns of the prediction dataframe:**
* `date`: The forecast target date (YYYY-MM-DD).
* `pred`: The median (50th percentile) prediction.
* `lower_*` and `upper_*`: These form your prediction intervals. For example, lower_95 (2.5th percentile) and upper_95 (97.5th percentile) create your 95% confidence interval.

Also, to be accepted by the platform, your prediction data must strictly adhere to the following rules:

* **Sunday dates**: The prediction date must fall on a Sunday (for weekly predictions) to match the platform's validation data.
* **Continuous dates**: There can be no gaps in your sequence of dates.
* **Challenge Timeframe**: Your predictions must cover all dates between Epidemiological Week (EW) 41 of the previous year and EW 40 of the target year.
* **Positive Values**: All prediction values must be 0 or greater (no negative numbers).
* **Nested Intervals**: Your intervals must make logical, mathematical sense. The values must increase sequentially exactly like this:
`lower_95` ≤ `lower_90` ≤ `lower_80` ≤ `lower_50` ≤ `pred` ≤ `upper_50` ≤ `upper_80` ≤ `upper_90` ≤ `upper_95`

Import the necessary packages:

In [1]:
import pandas as pd
from mosqlient import upload_prediction

The code below is used because the `api_key` is stored in a .env file. The `api_key` is required to submit predictions to the platform.

In [3]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv('Demo Notebooks/')

# Access the environment variables
api_key = os.getenv('API_KEY')

# Load the example prediction: 

The prediction below is a prediction from the last IMDC edition. It will be used here as an example of how submit a prediction.

In [4]:
prediction = pd.read_csv('example_prediction.csv')

prediction.head()

,date,pred,lower_50,upper_50,lower_80,upper_80,lower_90,upper_90,lower_95,upper_95
0,2025-10-05,263.135004,185.824100,363.016508,141.675773,485.086789,122.223771,607.371635,106.814960,765.932579
1,2025-10-12,305.696029,213.799311,441.210066,172.507355,622.408253,139.120725,758.241640,120.084934,969.836429
2,2025-10-19,266.370121,201.664501,386.144198,148.159385,542.579301,121.068139,697.049694,102.715650,807.432105
3,2025-10-26,352.163057,258.439042,492.270911,189.347334,697.579361,165.452554,924.080562,138.716489,1179.654947
4,2025-11-02,460.307748,336.501200,635.626532,255.112736,865.532747,219.084530,1040.971106,195.849099,1288.596162


#### Upload prediction: 

Fill the necessary parameters: 

In [5]:
repository = 'eduardocorrearaujo/3rd_imdc_emap_lstm' # fill with your repository name 
description = 'Example of submitting predictions in the Python tutorial'
commit = 'cafe6f3aa759fcdd569952eeeea462814870d678'
disease = 'A90' # dengue prediction
adm_level = 1 # state level prediction
adm_1 = 33 # example for RJ
adm_2 = None 
case_definition = 'probable' # The IMDC uses probable cases 
published = True

Run the code below to registry the forecast the prediction:

In [6]:

res = upload_prediction(
                api_key = api_key,
                repository = repository,
                description = description,
                commit = commit,
                disease=disease,
                case_definition=case_definition,
                adm_level = adm_level,
                adm_1 = adm_1,
                published = published,
                prediction = prediction
)

print(res)

{"id":5462,"model":{"id":39,"repository":"eduardocorrearaujo/3rd_imdc_emap_lstm","description":null,"category":"quantitative","time_resolution":"week","sprint":2026,"predictions_count":217,"active":true,"created_at":"2026-03-30","last_update":"2026-03-30"},"disease":"A90","commit":"cafe6f3aa759fcdd569952eeeea462814870d678","description":"Example of submitting predictions in the Python tutorial","case_definition":"probable","published":true,"start_date":null,"end_date":null,"scores":{},"adm_level":1,"adm_0":"BRA","adm_1":33,"adm_2":null,"adm_3":null,"data":[]}


After saving, it will appear in the “Predictions” tab of your model: 

![](../img/example_submission.png)


For more details check the [mosqlient documentation](https://mosqlimate-client.readthedocs.io/en/latest/tutorials/API/registry/). If you run into dificulties, please reach out fo help at our [discord server](https://discord.gg/yqtgW4TC).